# Notebook 2 — Semantic Exploration (CountVectorizer + Sentence Embeddings + UMAP)

> Goal: prepare reproducible data for semantic analysis, extract interpretable vocabulary by class, and visualize 2D structure with UMAP.

## 1) Environment setup and dependencies
Install/import libraries, set random seed, define paths, and configure visualization/logging options.

In [4]:
import logging
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
import umap.umap_ as umap
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 180)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from political_bias_api.core.paths import PROCESSED_DIR, USERS_CSV

ARTIFACTS_DIR = PROCESSED_DIR / "semantic_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts dir: {ARTIFACTS_DIR}")

Artifacts dir: /Users/sandrasanchezp/projects/nlp-political-bias-api/data/processed/semantic_artifacts


## 2) Data loading and schema validation (user-level only)
Read and validate only the user-level dataset in this phase. Post-level data will be added in a later section/iteration.

In [5]:
required_user_cols = {"user_id", "label", "n_posts", "text"}

users_df = pd.read_csv(USERS_CSV)
print(f"Users dataset: {USERS_CSV}")
print(f"Shape: {users_df.shape}")

missing_user_cols = required_user_cols - set(users_df.columns)
assert not missing_user_cols, f"Missing user-level columns: {missing_user_cols}"

display(users_df.dtypes.to_frame("dtype"))
display((users_df.isna().mean() * 100).sort_values(ascending=False).to_frame("missing_%"))
print("Duplicated user_id:", int(users_df["user_id"].duplicated().sum()))
print("Post-level loading deferred to a later notebook section.")

Users dataset: /Users/sandrasanchezp/projects/nlp-political-bias-api/data/processed/reddit_users_text_label.csv
Shape: (4150, 4)


,dtype
user_id,object
label,int64
n_posts,int64
text,object


,missing_%
user_id,0.0
label,0.0
n_posts,0.0
text,0.0


Duplicated user_id: 0
Post-level loading deferred to a later notebook section.


## 3) Initial cleaning and transformation (user-level only)
Create a clean user-level working dataset for downstream semantic analysis.

In [ ]:
def clean_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

MAX_USERS = 4000

work_df = users_df[["user_id", "label", "n_posts", "text"]].copy()
work_df["granularity"] = "user"
work_df["text"] = clean_text(work_df["text"])
work_df = work_df[work_df["text"].str.len() > 0].copy()

if len(work_df) > MAX_USERS:
    n_classes = work_df["label"].nunique()
    per_class = max(1, MAX_USERS // n_classes)
    work_df = (
        work_df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_class), random_state=SEED))
        .reset_index(drop=True)
    )

work_df["text_len_chars"] = work_df["text"].str.len()
work_df["text_len_words"] = work_df["text"].str.split().str.len()

print(f"Working dataset shape: {work_df.shape}")
print("Granularity:", work_df["granularity"].iloc[0])
display(work_df.head(3))